[◀ Notebook 06: Data Structures](06_data_structures.ipynb) | [Table of Contents](TOC.ipynb) | [Notebook 08: Exception Handling ▶](08_exception_handling.ipynb)

# Notebook 07: Functions and Scoping

This notebook explores the execution model of Python functions, lexical scoping namespaces, variable scope bindings (`nonlocal` and `global`), strict parameter layouts, and the underlying mechanics of closures.

Reference: [Defining Functions](https://docs.python.org/3/tutorial/controlflow.html#defining-functions)


## 1. Function Definition Mechanics

In Python, the `def` statement is an **executable statement**. When evaluated, it compiles the function body and binds a new function object to the function name in the local namespace.

```python
def my_func():
    return 42
```

This binds `my_func` to a function object. Since functions are first-class objects, they can be assigned to other variables, passed as arguments, or dynamically inspected.


In [ ]:
def greet(name):
    return f"Hello, {name}!"

print("Function type:", type(greet))
print("Function docstring attribute:", greet.__doc__)

# Assign function to a variable
salute = greet
print(salute("World"))


## 2. The LEGB Scoping Rule & Compiler Name Binding

Python resolves names using the **LEGB** search order:
1. **L**ocal: Names assigned within a function (and not declared global/nonlocal).
2. **E**nclosing: Names in local scopes of enclosing functions (outer closures).
3. **G**lobal: Names assigned at the top-level of the module.
4. **B**uilt-in: Names pre-defined in Python (e.g. `print`, `ValueError`).

### Compiler Binding Mechanics: `UnboundLocalError`
A critical execution mechanic is that **Python compiles variable scopes at compilation time**, not at runtime. 
If a variable is assigned a value anywhere inside a function, Python marks that name as *local* to the function's namespace. 
If you attempt to read that variable *before* the assignment line executes, Python raises an `UnboundLocalError` rather than looking it up in the global scope.


In [ ]:
x = 10

def attempt_shadow():
    try:
        # Even though x is global, Python flags 'x' as local to this function
        # because it is assigned to below (x = 20).
        print("Trying to print x:", x)  
        x = 20
    except UnboundLocalError as e:
        print(f"Caught UnboundLocalError: {e}")

attempt_shadow()


### Using `global` and `nonlocal`
- `global x`: Declares that `x` belongs to the module-level (global) namespace.
- `nonlocal x` (introduced in PEP 3101): Declares that `x` belongs to the nearest enclosing scope (excluding the global scope).


In [ ]:
count = 0

def increment_global():
    global count
    count += 1

increment_global()
print("Global count:", count)

def outer():
    state = "initial"
    def inner():
        nonlocal state
        state = "modified"
    inner()
    print("Outer state after inner running:", state)

outer()


## 3. Strict Parameter Layouts and Argument Mechanics

### The Mutable Default Argument Trap
Default parameter values are **evaluated once when the function is defined**, not on each function call. If you use a mutable default value (like a list or dict), that single object is shared across all subsequent calls that omit the parameter.


In [ ]:
def append_to(element, target_list=[]):
    target_list.append(element)
    return target_list

print(append_to(1))  # Returns [1]
print(append_to(2))  # Returns [1, 2] ! (Shared target_list object)

# Correct pattern:
def append_to_safe(element, target_list=None):
    if target_list is None:
        target_list = []
    target_list.append(element)
    return target_list

print(append_to_safe(1))  # [1]
print(append_to_safe(2))  # [2]


### Positional-Only and Keyword-Only Parameters
Modern Python allows strict signature limits to separate arguments:
- **Positional-Only (`/`)** (PEP 570): Parameters before `/` must be passed positionally and cannot be passed as keywords.
- **Keyword-Only (`*`)** (PEP 3102): Parameters after `*` must be passed using keyword arguments.


In [ ]:
def strict_api(pos_only, /, positional_or_keyword, *, kw_only):
    print(f"pos_only={pos_only}, mid={positional_or_keyword}, kw_only={kw_only}")

# Valid invocation:
strict_api(1, 2, kw_only=3)
strict_api(1, positional_or_keyword=2, kw_only=3)

# Invalid calls (uncomment to see exceptions):
# strict_api(pos_only=1, positional_or_keyword=2, kw_only=3)  # TypeError
# strict_api(1, 2, 3)                                         # TypeError


## 4. Closures and Inner Workings

A **closure** is a function object that retains bindings to variables in its enclosing lexical scope even after that enclosing scope has finished executing. CPython implements this using **cell objects** stored in the function's `__closure__` attribute.

### Shared Enclosing Scopes and State Sharing in Sibling Closures
When a single outer function defines multiple inner functions (referred to as **sibling closures**), and those inner functions access the same variables from the outer function's enclosing scope, they **share the exact same state**.

#### How State Sharing Works
In Python, closures do not copy the *values* of variables in the enclosing scope. Instead, they reference the variables themselves. Under the hood in CPython, this is implemented using `cell` objects. A cell object is a container that holds a reference to a Python object.
- When the outer function compiles, the compiler detects that a variable is accessed by inner functions.
- The compiler allocates a `cell` object in the outer function's stack frame to hold that variable.
- Each inner function's code object is created with a `__closure__` tuple containing a reference to the **exact same `cell` object**.
- Because all sibling closures reference the same `cell` object, any mutations made to the variable within one closure are immediately visible to all other sibling closures.

#### Detailed Code Example: Sibling Mutators
Below is an example showing how sibling closures (increment and decrement) modify and read a shared state variable (`state`):

```python
def create_state_manager(initial_state: int):
    state = initial_state  # Shared enclosing state

    def increment(amount: int = 1) -> int:
        nonlocal state
        state += amount
        return state

    def decrement(amount: int = 1) -> int:
        nonlocal state
        state -= amount
        return state

    def get_state() -> int:
        return state

    return increment, decrement, get_state

inc, dec, get_val = create_state_manager(10)

# The enclosing variable 'state' is shared between all three closures:
print(get_val())  # 10
inc(5)            # Mutates 'state' in the shared cell to 15
print(get_val())  # 15
dec(2)            # Mutates the same 'state' in the shared cell to 13
print(get_val())  # 13

# We can inspect the closures to verify they point to the exact same cell object:
print(inc.__closure__[0] is dec.__closure__[0])  # True
print(inc.__closure__[0] is get_val.__closure__[0])  # True
```

This model enables object-like state encapsulation purely using closures, without resorting to classes or global state.


In [ ]:
def make_multiplier(factor):
    def multiply(x):
        return x * factor
    return multiply

times_three = make_multiplier(3)
print("Result:", times_three(10))

# Inspect the closure cell
print("Closure attribute:", times_three.__closure__)
print("Cell contents:", times_three.__closure__[0].cell_contents)

# Sibling closures sharing state through local cell bindings
def make_sibling_mutators(initial_state):
    state = initial_state
    
    def increment():
        nonlocal state
        state += 1
        return state
        
    def decrement():
        nonlocal state
        state -= 1
        return state
        
    return increment, decrement

inc, dec = make_sibling_mutators(10)
print("\nSibling Closures state mutators:")
print("inc() call 1:", inc())
print("inc() call 2:", inc())
print("dec() call 1:", dec())

# Verify that both inc and dec closures share the exact same cell object
print("inc closure cell ID:", id(inc.__closure__[0]))
print("dec closure cell ID:", id(dec.__closure__[0]))


## 5. Coding Challenges

Complete the following exercises to test your understanding.


### Challenge 1: Scoping Bug Fix

Fix the code below so that calls to the returned `tracker` function successfully update the running total stored in `total` without throwing an `UnboundLocalError`.


In [ ]:
def make_tracker():
    """
    Returns a function that tracks a running total.
    Each call to the returned function adds the value passed to the running total and returns it.
    """
    total = 0
    def tracker(value):
        # Fix this function so that 'total' is correctly updated in the enclosing scope
        total += value
        return total
    return tracker


In [ ]:
# Run this cell to verify your implementation
try:
    my_tracker = make_tracker()
    assert my_tracker(10) == 10, "Failed on first add"
    assert my_tracker(5) == 15, "Failed on second add"
    assert my_tracker(-2) == 13, "Failed on negative values"
    print("🎉 Challenge 1 Passed Successfully!")
except Exception as e:
    print(f"❌ Test Failed: {type(e).__name__}: {e}")


### Challenge 2: Stateful Counter Closure

Implement `create_counter(initial_value)` which returns three functions: `(increment, decrement, get_value)`.
- `increment(n=1)` increases the inner state by `n` and returns the new value.
- `decrement(n=1)` decreases the inner state by `n` and returns the new value.
- `get_value()` returns the current state value without modifying it.

**Constraint**: Use closures and lexical variable bindings. Do not store the state on a global variable or in a custom class.


In [ ]:
def create_counter(initial_value: int = 0):
    """
    Creates a stateful counter.
    Returns a tuple: (increment_func, decrement_func, get_value_func)
    """
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
# Run this cell to verify your implementation
try:
    inc, dec, get_val = create_counter(10)
    
    assert get_val() == 10, "Initial value mismatch"
    assert inc() == 11, "Default increment failed"
    assert inc(4) == 15, "Custom increment failed"
    assert dec() == 14, "Default decrement failed"
    assert dec(5) == 9, "Custom decrement failed"
    assert get_val() == 9, "State corruption detected"
    
    # Test independent state isolation
    inc2, dec2, get_val2 = create_counter(100)
    assert get_val2() == 100
    assert get_val() == 9, "State leakage between closures!"
    
    print("🎉 Challenge 2 Passed Successfully!")
except AssertionError as e:
    print(f"❌ Test Failed: {e}")


### Challenge 3: Strict API Signature Design

Implement a user registration signature function named `register_user` satisfying these exact parameters:
1. `username` and `email` must be **positional-only** required arguments.
2. `role` (defaults to `'user'`) and `is_active` (defaults to `True`) must be **keyword-only** arguments.

The function should return a dictionary containing these elements, e.g.:
`{'username': username, 'email': email, 'role': role, 'is_active': is_active}`


In [ ]:
# Implement register_user with the strict signature specified above
# YOUR CODE HERE
def register_user(*args, **kwargs):
    raise NotImplementedError()


In [ ]:
# Run this cell to verify your implementation
try:
    # Happy Path
    u1 = register_user("alice", "alice@test.com")
    assert u1 == {"username": "alice", "email": "alice@test.com", "role": "user", "is_active": True}
    
    u2 = register_user("bob", "bob@test.com", role="admin", is_active=False)
    assert u2 == {"username": "bob", "email": "bob@test.com", "role": "admin", "is_active": False}
    
    # Check that username/email cannot be passed by keyword
    try:
        register_user(username="charlie", email="charlie@test.com")
        raise Exception("Allowed positional-only args to be passed by keyword!")
    except TypeError:
        pass # Correct!
        
    # Check that role/is_active cannot be passed positionally
    try:
        register_user("charlie", "charlie@test.com", "admin")
        raise Exception("Allowed keyword-only args to be passed positionally!")
    except TypeError:
        pass # Correct!
        
    print("🎉 Challenge 3 Passed Successfully!")
except AssertionError as e:
    print(f"❌ Test Failed: {e}")
except Exception as e:
    print(f"❌ Error in test execution: {e}")


[◀ Notebook 06: Data Structures](06_data_structures.ipynb) | [Table of Contents](TOC.ipynb) | [Notebook 08: Exception Handling ▶](08_exception_handling.ipynb)